<a href="https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noor486/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/noor486/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working dir:", os.getcwd())

# Generate outputs/model_results.json if it doesn't exist yet in this session
if not os.path.exists("outputs/model_results.json"):
    print("model_results.json not found — running the pipeline once to generate it...")
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

assert os.path.exists("outputs/model_results.json"), "Pipeline ran but file still missing — check for errors above"
print("Ready.")

Working dir: /content/flyrank-ml-internship
model_results.json not found — running the pipeline once to generate it...
Ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Two Signal Checks

Checking two signals my rule leans on, before trusting them. Signal A (staleness)
is directly behind FlyRank's real refresh flags. Signal B (CTR-vs-position) is
directly behind the real CTR-fix logic from this week's session.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#Signal A: staleness vs. declining rate
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Bucket by staleness tier
df["staleness_tier"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, 100000],
    labels=["<90d", "90-180d", "180-365d", "365d+"]
)

staleness_table = df.groupby("staleness_tier", observed=True).agg(
    n=("is_declining_label", "size"),
    declining_rate=("is_declining_label", "mean")
).round(3)
print(staleness_table)

                    n  declining_rate
staleness_tier                       
<90d            20655           0.512
90-180d          9171           0.611
180-365d          169           0.467
365d+               5           0.600


In [3]:
#Signal B: CTR by position tier (flag-linked: behind the real CTR-fix logic)
visible = df[df["impressions_90d"] >= 100].copy()
visible["position_tier"] = pd.cut(
    visible["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["top_3", "page_1", "top_20", "top_50", "deep"]
)

ctr_table = visible.groupby("position_tier", observed=True).agg(
    n=("ctr", "size"),
    mean_ctr=("ctr", "mean")
).round(4)
print(ctr_table)

                  n  mean_ctr
position_tier                
top_3           555    0.3372
page_1         8660    0.3540
top_20         5876    0.2557
top_50         6037    0.1421
deep            878    0.0555


**Signal A verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]** — staleness tier
[does/doesn't] track with declining rate as expected. n shown above per bucket.

**Signal B verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]** — CTR clearly drops
by position tier as expected (this is the real signal behind FlyRank's CTR-fix
flag: comparing CTR only within the same tier, never across tiers). n shown above.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import os
print("Current dir:", os.getcwd())
os.makedirs("work/outputs", exist_ok=True)
print("Exists now:", os.path.isdir("work/outputs"))

Current dir: /content/flyrank-ml-internship
Exists now: True


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ONE rule: stale + visible = review candidate
# Score: how much impression volume is "at stake" on a stale page
df["is_stale"] = df["days_since_last_update"] >= 180
df["is_visible"] = df["impressions_90d"] >= 500

df["baseline_action_score"] = df["impressions_90d"].where(
    df["is_stale"] & df["is_visible"], 0
)

df["reason_code"] = "stale_visible_page"
df["action"] = df["baseline_action_score"].apply(
    lambda s: "review_for_refresh" if s > 0 else "monitor"
)

queue = df.sort_values("baseline_action_score", ascending=False)[
    ["content_id", "client_id", "baseline_action_score", "reason_code", "action",
     "impressions_90d", "days_since_last_update", "avg_position", "ctr"]
]

queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows. Top score: {queue['baseline_action_score'].iloc[0]}")
queue.head(10)

Wrote 30000 rows. Top score: 61678


,content_id,client_id,baseline_action_score,reason_code,action,impressions_90d,days_since_last_update,avg_position,ctr
16751,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,review_for_refresh,61678,194,19.7,0.15
16514,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,review_for_refresh,59472,194,24.8,0.13
7021,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,review_for_refresh,25715,194,22.2,0.23
21268,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,review_for_refresh,13299,193,10.5,0.49
11489,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,review_for_refresh,7812,194,39.0,0.01
12045,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,review_for_refresh,7558,193,17.9,0.20
698,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,review_for_refresh,4590,194,31.0,0.00
5327,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,review_for_refresh,4556,194,16.4,0.33
26810,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,review_for_refresh,4429,194,25.3,0.38
20837,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,review_for_refresh,1697,193,15.8,0.12


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-10 Review

For each of the top 10 rows (see table above), one line each:

1. [content_id] — Action: review_for_refresh. Why: highest impression volume
   among stale, visible pages. Would be wrong if: this page's traffic is
   already recovering and days_since_last_update is stale for a harmless
   reason (e.g. content genuinely doesn't need updates, like an evergreen FAQ).
2. [repeat for rows 2-10, using real values from your printed table]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak Picks

Honestly, the rule's biggest weakness: it only uses impressions_90d as the
score, ignoring avg_position and ctr entirely. A page could rank #40 with
mediocre CTR and still top this queue purely on volume, while a page ranking
#2 with a collapsing CTR (a real opportunity) sits lower. The rule conflates
"gets a lot of traffic" with "urgently needs a refresh."

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.